# 🚀 YOLOv8 + WandB Sweep (MLOps Demo)

이 노트북은 **Google Colab** 환경에서 YOLOv8과 Weights & Biases(WandB) Sweep을 활용한  
하이퍼파라미터 자동 최적화 파이프라인을 실습합니다.

---
### 📋 실행 순서
1. 환경 설치 (GPU 확인 → 패키지 설치)
2. 프로젝트 파일 준비 (GitHub 클론)
3. WandB 로그인
4. Baseline 학습
5. Sweep 실행
6. 결과 분석

> ⚠️ **GPU 필수**: 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## ① 환경 설정

In [1]:
# GPU 확인
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU가 없습니다. 런타임 > 런타임 유형 변경 > GPU 선택 후 다시 실행하세요.")

CUDA available: True
GPU: Tesla T4


In [2]:
# 필수 패키지 설치
%pip install -q ultralytics wandb
print("✅ 패키지 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.3 MB/s eta 0:00:00
✅ 패키지 설치 완료


## ② 프로젝트 파일 준비 (GitHub 클론)

In [3]:
import os, subprocess, sys

REPO_URL = "https://github.com/jspark9703/mlops_demo.git"  # ← 필요시 수정
WORK_DIR = "/content/mlops_demo"

if not os.path.exists(WORK_DIR):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
    print("✅ 클론 완료")
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)
    print("✅ 최신 코드로 업데이트")

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print(f"📂 작업 디렉토리: {os.getcwd()}")

✅ 클론 완료
📂 작업 디렉토리: /content/mlops_demo


## ③ WandB 로그인

[https://wandb.ai/authorize](https://wandb.ai/authorize) 에서 API 키를 복사해 붙여넣으세요.

In [4]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jspark9703 (jspark9703-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## ④ Baseline 학습

Sweep 비교를 위한 기준 모델을 먼저 학습합니다.

In [5]:
import sys, os

WORK_DIR = "/content/mlops_demo"
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# 모듈 캐시 초기화 (재실행 시 최신 파일 반영)
if "train_baseline" in sys.modules:
    del sys.modules["train_baseline"]

from train_baseline import main as run_baseline

print("🏁 Baseline 학습 시작...")
results = run_baseline()
print("✅ Baseline 학습 완료!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🏁 Baseline 학습 시작...


Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0,

✅ Baseline 학습 완료!


## ⑤ WandB Sweep 실행

Bayesian 최적화로 하이퍼파라미터를 자동 탐색합니다.

| 파라미터 | 탐색 범위 |
|---|---|
| `batch` | 8, 16, 32 |
| `epochs` | 10, 15, 20 |
| `imgsz` | 320, 480, 640 |
| `lr0` | 0.0001 ~ 0.01 (uniform) |
| `weight_decay` | 0.0001 ~ 0.001 (uniform) |

In [6]:
import wandb, yaml, sys, os

WORK_DIR = "/content/mlops_demo"
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# ── Baseline 학습 이후 잔여 wandb 상태 초기화 ──────────────────────
# wandb.init() 완료 후 내부 상태가 남아 wandb.sweep()이 블로킹되는
# 현상을 방지합니다. 이미 finish된 경우 예외를 무시합니다.
try:
    wandb.finish()
except Exception:
    pass

# ── WandB entity 설정 ─────────────────────────────────────────────
# entity를 명시하지 않으면 wandb.sweep()이 대화형 입력을 기다려
# 셀이 pending 상태로 멈출 수 있습니다.
# 아래 값을 본인의 WandB entity(계정명)로 수정하세요.
# wandb.ai 접속 → 우상단 프로필 → 'Copy Entity Name'
WANDB_ENTITY = "jspark9703-"  # ← 본인 entity로 수정

# ── Sweep 설정 (Python dict, 파일 의존 없음) ──────────────────────
sweep_config = {
    "method": "bayes",
    "metric": {
        "goal": "maximize",
        "name": "metrics/mAP50-95(B)"
    },
    "parameters": {
        "batch":        {"values": [8, 16, 32]},
        "epochs":       {"values": [10, 15, 20]},
        "imgsz":        {"values": [320, 480, 640]},
        "lr0":          {"distribution": "uniform", "min": 0.0001, "max": 0.01},
        "weight_decay": {"distribution": "uniform", "min": 0.0001, "max": 0.001},
    }
}

print("📋 Sweep 설정:")
print(yaml.dump(sweep_config, default_flow_style=False))

# entity를 명시해야 대화형 확인 없이 즉시 Sweep이 생성됩니다.
sweep_id = wandb.sweep(
    sweep=sweep_config,
    project="mlops_demo_yolo",
    entity=WANDB_ENTITY,
)
print(f"\n✅ Sweep 생성 완료!")
print(f"🔗 Sweep ID: {sweep_id}")
print(f"🌐 https://wandb.ai/{WANDB_ENTITY}/mlops_demo_yolo/sweeps/{sweep_id}")

📋 Sweep 설정:
method: bayes
metric:
  goal: maximize
  name: metrics/mAP50-95(B)
parameters:
  batch:
    values:
    - 8
    - 16
    - 32
  epochs:
    values:
    - 10
    - 15
    - 20
  imgsz:
    values:
    - 320
    - 480
    - 640
  lr0:
    distribution: uniform
    max: 0.01
    min: 0.0001
  weight_decay:
    distribution: uniform
    max: 0.001
    min: 0.0001

Create sweep with ID: kjdjprfe
Sweep URL: https://wandb.ai/jspark9703-/mlops_demo_yolo/sweeps/kjdjprfe

✅ Sweep 생성 완료!
🔗 Sweep ID: kjdjprfe
🌐 https://wandb.ai/jspark9703-/mlops_demo_yolo/sweeps/kjdjprfe


In [7]:
import wandb, sys

# 모듈 캐시 초기화
if "train_sweep" in sys.modules:
    del sys.modules["train_sweep"]

from train_sweep import main as sweep_agent_fn

# ── 실행할 Sweep 횟수 ─────────────────────────────────────────────
# Colab 무료 T4 기준: 에포크 10 × 3회 ≈ 30~60분 소요 예상
SWEEP_COUNT = 3  # ← 필요에 따라 수정 (최대 10 권장)

print(f"🤖 Sweep Agent 시작 (총 {SWEEP_COUNT}회 실행)...")
wandb.agent(
    sweep_id,
    function=sweep_agent_fn,
    count=SWEEP_COUNT
)
print("🎉 모든 Sweep 실행 완료!")

🤖 Sweep Agent 시작 (총 3회 실행)...


wandb: Agent Starting Run: npcmm41j with config:
wandb: 	batch: 8
wandb: 	epochs: 10
wandb: 	imgsz: 320
wandb: 	lr0: 0.0038071572148445985
wandb: 	weight_decay: 0.0006360555052530842
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0038071572148445985, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

epoch,▁▂▃▃▄▅▆▆▇██
metrics/mAP50(B),▁▂▃▄▆▆▇████
metrics/mAP50-95(B),▁▁▁▂▄▅▆████
metrics/precision(B),▁▂▄▅██▆▂▄█▆
metrics/recall(B),▁▁▂▂▃▃▅█▇▇▇
val/box_loss,▄▅▆██▆▄▁▁▁
val/cls_loss,█▆▅▅▄▄▃▂▂▁
val/dfl_loss,█▇▅▆▆▄▃▁▁▂
epoch,9
metrics/mAP50(B),0.58146
metrics/mAP50-95(B),0.42473


wandb: Agent Starting Run: r6t03pgi with config:
wandb: 	batch: 8
wandb: 	epochs: 15
wandb: 	imgsz: 640
wandb: 	lr0: 0.0032552798780202336
wandb: 	weight_decay: 0.0004531596984942305
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇██
metrics/mAP50(B),▁▂▃▄▄▅▆▆▇▇▇█████
metrics/mAP50-95(B),▁▂▃▄▄▅▅▆▇▇▇▇████
metrics/precision(B),▁▂▁▂▅▆██▇▇▆▅▇▆▆▆
metrics/recall(B),▁▂▄▆▅▆▆▆▇▇▇█▇█▇█
val/box_loss,█▇▆▅▄▄▄▄▃▃▃▃▂▁▁
val/cls_loss,█▇▆▅▅▄▄▃▃▃▂▂▁▁▁
val/dfl_loss,██▇▆▅▅▄▄▄▄▃▃▂▁▁
epoch,14
metrics/mAP50(B),0.74226
metrics/mAP50-95(B),0.5556


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: a1x89sne with config:
wandb: 	batch: 8
wandb: 	epochs: 10
wandb: 	imgsz: 640
wandb: 	lr0: 0.002821447021790282
wandb: 	weight_decay: 0.0002239417048420819
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002821447021790282, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, 

epoch,▁▂▃▃▄▅▆▆▇██
metrics/mAP50(B),▁▂▄▅▆▆▇▇███
metrics/mAP50-95(B),▁▂▄▄▆▆▇▇███
metrics/precision(B),▃▅█▆█▅▃▃▂▁▁
metrics/recall(B),▁▁▃▅▅▆▆▇▇██
val/box_loss,█▆▆▅▄▃▂▂▁▁
val/cls_loss,█▇▆▄▃▂▂▁▁▁
val/dfl_loss,█▆▅▅▄▄▃▂▁▁
epoch,9
metrics/mAP50(B),0.70297
metrics/mAP50-95(B),0.52586


🎉 모든 Sweep 실행 완료!


## ⑥ 결과 분석

In [8]:
import wandb
import pandas as pd

api = wandb.Api()

# sweep_id / WANDB_ENTITY 는 위 셀에서 정의한 값을 사용합니다.
# 별도 실행 시 아래처럼 직접 지정하세요:
# WANDB_ENTITY = "jspark9703-"
# sweep_id = "your-sweep-id"

sweep = api.sweep(f"{WANDB_ENTITY}/mlops_demo_yolo/{sweep_id}")
runs = sweep.runs

records = []
for run in runs:
    record = {
        "run_name": run.name,
        "state":    run.state,
        "mAP50-95": run.summary.get("metrics/mAP50-95(B)", None),
        "mAP50":    run.summary.get("metrics/mAP50(B)",    None),
    }
    record.update(run.config)
    records.append(record)

df = pd.DataFrame(records)
df_sorted = df.sort_values("mAP50-95", ascending=False).reset_index(drop=True)

print("🏆 Sweep 결과 (mAP50-95 기준 내림차순):")
display(df_sorted)

🏆 Sweep 결과 (mAP50-95 기준 내림차순):


,run_name,state,mAP50-95,mAP50,lr0,batch,imgsz,epochs,weight_decay
0,good-sweep-2,finished,0.555596,0.742263,0.003255,8,640,15,0.000453
1,swept-sweep-3,finished,0.525861,0.702972,0.002821,8,640,10,0.000224
2,olive-sweep-1,finished,0.424734,0.581456,0.003807,8,320,10,0.000636


In [9]:
# 최적 하이퍼파라미터 요약 출력
if not df_sorted.empty and df_sorted["mAP50-95"].notna().any():
    best = df_sorted.iloc[0]
    print("🥇 최적 하이퍼파라미터 조합:")
    print(f"   Run Name    : {best['run_name']}")
    print(f"   mAP50-95    : {best['mAP50-95']:.4f}")
    print(f"   mAP50       : {best.get('mAP50', 'N/A')}")
    print(f"   lr0         : {best.get('lr0', 'N/A')}")
    print(f"   weight_decay: {best.get('weight_decay', 'N/A')}")
    print(f"   batch       : {best.get('batch', 'N/A')}")
    print(f"   epochs      : {best.get('epochs', 'N/A')}")
    print(f"   imgsz       : {best.get('imgsz', 'N/A')}")
else:
    print("⚠️ 완료된 run이 없거나 mAP50-95 값이 없습니다.")

🥇 최적 하이퍼파라미터 조합:
   Run Name    : good-sweep-2
   mAP50-95    : 0.5556
   mAP50       : 0.742262825611327
   lr0         : 0.0032552798780202336
   weight_decay: 0.0004531596984942305
   batch       : 8
   epochs      : 15
   imgsz       : 640
